# 装饰器（Decorator）##  学习目标- 理解装饰器的概念和原理- 掌握装饰器的基本用法（@语法糖）- 能写带参数装饰器和类装饰器- 理解 `functools.wraps` 的作用- 能在实战中使用装饰器---## 1️⃣ 引入：函数是一等公民### 核心概念回顾在Python中，**函数是一等公民（first-class citizen）**，意味着：- 函数可以赋值给变量- 函数可以作为参数传递- 函数可以嵌套定义- 函数可以作为返回值（闭包）### 示例1：函数赋值给变量

In [1]:
# 示例1：函数赋值给变量def func(message):print('Got a message: {}'.format(message))send_message = func  # 函数赋值给变量send_message('hello world')  # 调用变量，相当于调用函数

Got a message: hello world


### 示例2：函数作为参数```pythondef get_message(message):return 'Got a message: ' + messagedef root_call(func, message):print(func(message))root_call(get_message, 'hello world')```**输出**：```Got a message: hello world```### 示例3：函数嵌套定义```pythondef func(message):def get_message(message):print('Got a message: {}'.format(message))return get_message(message)func('hello world')```### 示例4：函数作为返回值（闭包）```pythondef func_closure():def get_message(message):print('Got a message: {}'.format(message))return get_message  # 返回函数对象send_message = func_closure()send_message('hello world')```

In [ ]:
# 示例2-4代码演示# 示例2：函数作为参数def get_message(message):return 'Got a message: ' + messagedef root_call(func, message):print(func(message))print("示例2：函数作为参数")root_call(get_message, 'hello world')print()# 示例4：函数作为返回值（闭包）def func_closure():def get_message(message):print('Got a message: {}'.format(message))return get_messageprint("示例4：闭包")send_message = func_closure()send_message('hello world')

---## 2️⃣ 装饰器基础### 什么是装饰器？**装饰器（Decorator）**：在不修改原函数的情况下，为函数添加额外功能的包装器。**通俗理解**：- 就像给函数"穿衣服"- 原函数不变，但行为被"装饰"了### 简单的装饰器示例```pythondef my_decorator(func):def wrapper():print('wrapper of decorator')  # 添加的额外功能func()  # 调用原函数return wrapperdef greet():print('hello world')greet = my_decorator(greet)  # 手动装饰greet()```**输出**：```wrapper of decoratorhello world```

In [ ]:
# 简单装饰器演示def my_decorator(func):def wrapper():print('wrapper of decorator')func()return wrapperdef greet():print('hello world')print("手动装饰:")greet = my_decorator(greet)greet()

### @ 语法糖（推荐写法）Python提供了更简洁的语法：```python@my_decorator  # 等价于 greet = my_decorator(greet)def greet():print('hello world')greet()  # 现在调用的是 wrapper()```**优点**：- 更简洁- 可读性更好- 多个装饰器时更方便---## 3️⃣ 带参数的装饰器### 问题：原函数有参数怎么办？如果原函数有参数，装饰器需要能接受任意参数：```pythondef my_decorator(func):def wrapper(*args, **kwargs):  # 接受任意参数print('wrapper of decorator')func(*args, **kwargs)  # 传递给原函数return wrapper@my_decoratordef greet(message):print(message)greet('hello world')```**关键点**：- `*args`：接受任意位置参数- `**kwargs`：接受任意关键字参数- 这样装饰器就能装饰任意函数了

In [ ]:
# 带参数的装饰器演示def my_decorator(func):def wrapper(*args, **kwargs):print('wrapper of decorator')return func(*args, **kwargs)  # 注意：如果原函数有返回值，这里要returnreturn wrapper@my_decoratordef greet(message):print(message)greet('hello world')

---## 4️⃣ 带自定义参数的装饰器### 问题：装饰器本身需要参数怎么办？比如，我们想让装饰器执行多次：```pythondef repeat(num):  # 第一层：接受装饰器参数def my_decorator(func):  # 第二层：接受函数def wrapper(*args, **kwargs):  # 第三层：包装函数for i in range(num):print('wrapper of decorator')func(*args, **kwargs)return wrapperreturn my_decorator@repeat(4)  # 装饰器带参数def greet(message):print(message)greet('hello world')```**输出**：```wrapper of decoratorhello worldwrapper of decoratorhello world... (重复4次)```**三层嵌套结构**：```repeat(num)  # 第一层：装饰器工厂，返回装饰器└─> my_decorator(func)  # 第二层：装饰器，返回wrapper└─> wrapper(*args, **kwargs)  # 第三层：包装函数，调用原函数```

In [ ]:
# 带自定义参数的装饰器演示def repeat(num):def my_decorator(func):def wrapper(*args, **kwargs):for i in range(num):print(f'Execution {i+1}')func(*args, **kwargs)return wrapperreturn my_decorator@repeat(3)def greet(message):print(message)greet('hello world')

---## 5️⃣ `functools.wraps` 的重要性### 问题：装饰后原函数的元信息丢失```pythondef my_decorator(func):def wrapper(*args, **kwargs):print('wrapper')func(*args, **kwargs)return wrapper@my_decoratordef greet(message):'''这是一个问候函数'''print(message)print(greet.__name__)  # 输出：wrapper （不是greet！）print(greet.__doc__)   # 输出：None （文档字符串丢失！）```**问题**：装饰后，`greet` 变成了 `wrapper`，原函数的名字、文档字符串等元信息都丢失了！### 解决方案：`functools.wraps````pythonimport functoolsdef my_decorator(func):@functools.wraps(func)  # 保留原函数的元信息def wrapper(*args, **kwargs):print('wrapper')func(*args, **kwargs)return wrapper@my_decoratordef greet(message):'''这是一个问候函数'''print(message)print(greet.__name__)  # 输出：greet （正确！）print(greet.__doc__)   # 输出：这是一个问候函数 （正确！）```**最佳实践**：** always use `@functools.wraps(func)`！**

In [ ]:
# functools.wraps 演示import functoolsdef my_decorator(func):@functools.wraps(func)  # 保留元信息def wrapper(*args, **kwargs):print('wrapper')return func(*args, **kwargs)return wrapper@my_decoratordef greet(message):'''这是一个问候函数'''print(message)print("函数名:", greet.__name__)print("文档字符串:", greet.__doc__)

---## 6️⃣ 类装饰器### 概念类也可以作为装饰器，依赖于 `__call__()` 方法。```pythonclass Count:def __init__(self, func):self.func = funcself.num_calls = 0def __call__(self, *args, **kwargs):self.num_calls += 1print('num of calls is: {}'.format(self.num_calls))return self.func(*args, **kwargs)@Countdef example():print('hello world')example()  # 第1次调用example()  # 第2次调用```**输出**：```num of calls is: 1hello worldnum of calls is: 2hello world```**解析**：- `@Count` 创建了两个实例- 每次调用 `example()`，实际上是调用 `__call__()` 方法- `__call__()` 记录调用次数，然后调用原函数

In [ ]:
# 类装饰器演示class Count:def __init__(self, func):self.func = funcself.num_calls = 0def __call__(self, *args, **kwargs):self.num_calls += 1print('num of calls is: {}'.format(self.num_calls))return self.func(*args, **kwargs)@Countdef example():print('hello world')example()example()example()

---## 7️⃣ 装饰器的嵌套### 多个装饰器的执行顺序```python@decorator1@decorator2@decorator3def func():...```**执行顺序**：从**里到外**（从下到上）等价于：```pythonfunc = decorator1(decorator2(decorator3(func)))```### 示例：多个装饰器```pythonimport functoolsdef decorator1(func):@functools.wraps(func)def wrapper(*args, **kwargs):print('execute decorator1')return func(*args, **kwargs)return wrapperdef decorator2(func):@functools.wraps(func)def wrapper(*args, **kwargs):print('execute decorator2')return func(*args, **kwargs)return wrapper@decorator1@decorator2def greet(message):print(message)greet('hello world')```**输出**：```execute decorator1  ← 先执行 decorator1 的wrapperexecute decorator2  ← 再执行 decorator2 的wrapperhello world      ← 最后执行原函数```

In [ ]:
# 装饰器嵌套演示import functoolsdef decorator1(func):@functools.wraps(func)def wrapper(*args, **kwargs):print('execute decorator1')return func(*args, **kwargs)return wrapperdef decorator2(func):@functools.wraps(func)def wrapper(*args, **kwargs):print('execute decorator2')return func(*args, **kwargs)return wrapper@decorator1@decorator2def greet(message):print(message)greet('hello world')

---## 8️⃣ 实战应用：装饰器的常见场景### 1. 计时装饰器```pythonimport timeimport functoolsdef timer(func):@functools.wraps(func)def wrapper(*args, **kwargs):start = time.time()result = func(*args, **kwargs)end = time.time()print(f'{func.__name__} took {end - start:.2f} seconds')return resultreturn wrapper@timerdef slow_function():time.sleep(1)return 'done'slow_function()```### 2. 日志装饰器```pythonimport functoolsdef logger(func):@functools.wraps(func)def wrapper(*args, **kwargs):print(f'Calling {func.__name__} with args={args}, kwargs={kwargs}')result = func(*args, **kwargs)print(f'{func.__name__} returned {result}')return resultreturn wrapper@loggerdef add(a, b):return a + badd(3, 5)```### 3. 缓存装饰器（递归优化）```pythonimport functools@functools.lru_cache(maxsize=None)  # 内置缓存装饰器def fib(n):if n < 2:return nreturn fib(n-1) + fib(n-2)print(fib(50))  # 快速计算，因为缓存了中间结果```

In [ ]:
# 实战应用演示import timeimport functools# 1. 计时装饰器def timer(func):@functools.wraps(func)def wrapper(*args, **kwargs):start = time.time()result = func(*args, **kwargs)end = time.time()print(f'{func.__name__} took {end - start:.2f} seconds')return resultreturn wrapper@timerdef slow_function():time.sleep(1)return 'done'print("1. 计时装饰器:")slow_function()

---## 9️⃣ 常见错误与陷阱### 陷阱1：忘记使用 `functools.wraps````python# 错误示例def my_decorator(func):def wrapper(*args, **kwargs):  # 没有 @functools.wraps(func)return func(*args, **kwargs)return wrapper@my_decoratordef greet():'''文档字符串'''passprint(greet.__name__)  # 'wrapper' （错误！）```**改正**：加上 `@functools.wraps(func)`### 陷阱2：装饰器不返回函数值```python# 错误示例def my_decorator(func):@functools.wraps(func)def wrapper(*args, **kwargs):result = func(*args, **kwargs)  # 计算结果# 忘记 return result 了！return wrapper@my_decoratordef add(a, b):return a + bresult = add(3, 5)print(result)  # None （错误！）```**改正**：在 `wrapper` 中 `return func(*args, **kwargs)`---##  练习与自测###  基础自测（概念检查）- [ ] 能解释什么是装饰器- [ ] 能说出 `@decorator` 等价于什么代码- [ ] 能解释为什么需要 `functools.wraps`- [ ] 能说出装饰器嵌套的执行顺序- [ ] 能解释 `*args, **kwargs` 的作用###  代码练习#### 题目1：手写计时装饰器（⭐⭐）**题目**：写一个装饰器 `timer`，记录函数执行时间。**你的代码**：

In [ ]:
# 题目1：手写计时装饰器import timeimport functoolsdef timer(func):# 你的代码 herepass# 测试@timerdef slow_function():time.sleep(1)return 'done'slow_function()

#### 题目2：带参数装饰器（⭐⭐⭐）**题目**：写一个装饰器 `repeat(num)`，让函数重复执行 `num` 次。**你的代码**：

In [ ]:
# 题目2：带参数装饰器import functoolsdef repeat(num):# 你的代码 herepass# 测试@repeat(3)def greet(message):print(message)greet('hello')

---##  知识总结### 核心要点1. **装饰器是 callable 对象**，可以接受函数作为参数，并返回函数2. **@ 语法糖**，`@decorator` 等价于 `func = decorator(func)`3. **带参数装饰器是三层嵌套**：参数层 → 装饰器层 → wrapper层4. **总是使用 `functools.wraps`**，保留原函数元信息5. **装饰器嵌套**，执行顺序从里到外（从下到上）6. **类装饰器**，依赖于 `__call__()` 方法### 装饰器模板```pythonimport functoolsdef my_decorator(func):@functools.wraps(func)def wrapper(*args, **kwargs):# 执行前result = func(*args, **kwargs)# 执行后return resultreturn wrapper```### 与其他知识的关系- **前置知识**：函数、闭包、一等公民- **后续知识**：元类（metaclass）、描述符---##  扩展阅读1. **官方文档 - 装饰器**https://docs.python.org/3/glossary.html#term-decorator2. **内置装饰器**- `@property`：将方法转换为属性- `@staticmethod`：静态方法- `@classmethod`：类方法- `@functools.lru_cache`：缓存3. **相关面试题**- 手写装饰器（计时、日志、缓存）- 解释装饰器的执行原理- 多个装饰器的执行顺序---##  学习检查（学完自测）- [ ] 能不看资料，默写出装饰器模板- [ ] 能独立完成所有练习- [ ] 能讲给别人听（费曼学习法）- [ ] 能说出装饰器在实际项目中的应用场景---**Next**: 下一讲：生成器与迭代器 → 惰性计算与节省内存！